# K-Means（1967）

- **核心原理**：迭代更新群中心
- **變體**：K-Means++（聰明的初始化）、Mini-Batch K-Means（大資料）；K-Medoids、Kernel K-Means
- **推薦資料集**：Wholesale Customers（440 家批發商的客戶、6 類商品的年消費金額）

這是第一部分第一個**非監督學習**（unsupervised）的模型：資料**沒有標籤**，模型要自己找出「長得像的一群」。README 的每章標準流程照走，但有幾步的意思要調整：

| 步驟 | 監督學習 | 本章（分群） |
| --- | --- | --- |
| 2. Baseline | Dummy | 「全部一群」、「隨機分群」、「用 Channel 分群」 |
| 5. 調參 | CV 選超參數 | 選群數 $K$：elbow、silhouette、穩定度 |
| 6. 測試集評估 | 測試集只看一次 | 沒有測試集；用**內部指標**（silhouette 等）與**外部標籤**（Channel，訓練時沒用到）檢查 |
| 7. 診斷 | 看錯誤 | 看每一群的「長相」（群中心），判斷有沒有意義 |

K-Means 的想法很簡單：**每一群用一個中心點代表，每個點歸給最近的中心，中心移到它那群的平均** —— 重複到不再變動。這一章的重點：**這個迴圈為什麼一定會停、為什麼會停在「不好」的地方（初始化）、距離的定義（尺度、log）決定了一切、怎麼選 $K$，以及 K-Means 天生看不到哪些形狀的群。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A、3B、3C、4、5、7 節開頭另有各自的符號表或名詞表。

**資料**：每一列是一個客戶（餐廳、旅館、零售店……）。6 個消費欄位（年消費金額，貨幣單位 m.u.）：`Fresh`（生鮮）、`Milk`（乳製品）、`Grocery`（雜貨）、`Frozen`（冷凍）、`Detergents_Paper`（清潔用品與紙類）、`Delicassen`（熟食；UCI 原始欄名就是這個拼法）。另外兩欄 **`Channel`**（1 = Horeca：旅館／餐廳／咖啡廳、2 = Retail：零售）與 `Region`（1–3，地區）**不拿來分群**，只在事後檢查分出來的群有沒有意義。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $n$、$d$ | 樣本數、特徵數 | 440、6 |
| $X$ | 前處理後的資料（log 再標準化） | $(n, d)$ |
| $K$ | 群數 | — |
| $\mu_k$ | 第 $k$ 群的中心（centroid） | $(K, d)$ 的第 $k$ 列 |
| $c_i$ | 第 $i$ 個點被分到哪一群（0..K−1） | $(n,)$ |
| inertia $J$ | $\sum_i \lVert x_i - \mu_{c_i}\rVert^2$：每個點到自己群中心的平方距離總和（群內平方和，WCSS） | 純量 |
| silhouette $s_i$ | 第 $i$ 點「跟自己群的距離」vs「跟最近的別群的距離」，$\in [-1, 1]$，越大越好 | $(n,)$ |
| ARI | adjusted Rand index：兩種分群的一致程度；1 = 完全一樣、0 ≈ 隨機 | — |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第四章 | KNN（`01-classic-ml/04-knn`） | 平方距離矩陣 $\lVert a\rVert^2 + \lVert b\rVert^2 - 2a\cdot b$ 一次算完；縮放 = 決定每個特徵在距離裡的權重；單位不同時必須標準化。 |
| 第四章的 NearestCentroid | 同上，TODO 2 (2) | 每一類一個平均點、找最近的中心；假設每一類是「單一一團、球狀」。K-Means 就是**沒有標籤**的 NearestCentroid。 |
| 第十二章前置複習 1 | GBDT | 讓 $\sum_i (r_i - c)^2$ 最小的常數 $c$ 是平均。 |
| 第 15 章 GMM、第 14 章 DBSCAN | 下兩章 | GMM：軟分群、每群可以是不同形狀的橢圓；DBSCAN：依密度分群、可以是任意形狀、會把雜訊點挑出來。 |
| 第 16 章 PCA | | 把高維資料投影到變異最大的幾個方向；本章只拿來「畫圖」。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 平方距離**：$\lVert a - b\rVert^2 = \sum_j (a_j - b_j)^2$。

**2. 平均讓平方距離總和最小**（第十二章前置複習 1 的向量版）：給一群點 $x_1..x_m$，讓 $\sum_i \lVert x_i - \mu\rVert^2$ 最小的 $\mu$ 是它們的平均。例：一維 $(1, 2, 6)$ → $\mu = 3$，總和 $4 + 1 + 9 = 14$；換成 $\mu = 2$ → $1 + 0 + 16 = 17$，比較大。

**3. 坐標下降（coordinate descent）**：一個函數有兩組變數 $(c, \mu)$ 時，輪流「固定一組、把另一組調到最好」。每一步都不會讓函數變大 → 函數值單調下降；但只保證停在某個「兩邊都調不動」的點，**不保證是全域最小**。

**4. 機率正比於某個量的抽樣**：權重 $w_i \ge 0$，令 $p_i = w_i / \sum_j w_j$，`rng.choice(n, p=p)`。例：$w = (1, 0, 3)$ → 抽到第 2 個的機率 0.75、第 1 個 0（K-Means++ 用這個）。

**5. log 轉換**（第十二章前置複習 5）：右偏的金額取 $\log(1 + x)$ 之後比較對稱；log 的差 ≈ 倍數的差（花 100 vs 1,000 和 1,000 vs 10,000 在 log 上一樣遠）。

**6. 平均與標準化**：標準化 = 每欄減平均、除標準差（第三、四章）。

#### 程式

**7. 平方距離矩陣**（第四章 TODO 3）：
```python
D = (X**2).sum(axis=1)[:, None] + (C**2).sum(axis=1)[None, :] - 2 * X @ C.T   # (n, K)
```

**8. 每列的最小值位置**：`D.argmin(axis=1)` → $(n,)$；`D.min(axis=1)` → 最小值本身。

**9. 分組平均**：`X[labels == k].mean(axis=0)` = 第 $k$ 群的平均；某一群沒有任何點時，`.mean` 會給 `nan`（還會跳警告）。

**10. 每個點取「自己那一群」的中心**：`C[labels]` → $(n, d)$（fancy indexing：第 $i$ 列是 `C[labels[i]]`）。

**11. `pd.crosstab(a, b)`**：兩個分組的交叉次數表。

---
## 1. 資料與探索性資料分析（EDA）

觀察直接寫在每格程式最上面的註解（`# 觀察：`）。

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ml_dl_llm.data import load_wholesale_customers

df = load_wholesale_customers()
# 統一欄名：UCI 原始是 "Delicassen"、有些版本是 "Delicatessen"；大小寫也統一
rename = {c: c for c in df.columns}
for c in df.columns:
    if c.lower() in ("delicatessen", "delicassen"):
        rename[c] = "Delicassen"
    elif c.lower() == "detergents_paper":
        rename[c] = "Detergents_Paper"
    elif c.lower() in ("channel", "region", "fresh", "milk", "grocery", "frozen"):
        rename[c] = c.capitalize()
df = df.rename(columns=rename)
SPEND = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
print(df.shape)
display(df.head())
print("Channel：", df["Channel"].value_counts().sort_index().to_dict(), "（1 = Horeca、2 = Retail）")
print("Region ：", df["Region"].value_counts().sort_index().to_dict())

In [ ]:
# 消費金額的分布
# 觀察：（平均和中位數差多少？最大值是中位數的幾倍？取 log 之後呢？
#        不取 log 直接算距離，哪一欄會主導？—— 第四章：份量 ∝ 變異數）
display(df[SPEND].describe().T.round(0))
fig, axes = plt.subplots(2, 6, figsize=(18, 5))
for j, c in enumerate(SPEND):
    axes[0, j].hist(df[c], bins=40)
    axes[0, j].set_title(c)
    axes[1, j].hist(np.log1p(df[c]), bins=40, color="C1")
    axes[1, j].set_title(f"log1p({c})")
fig.tight_layout()
plt.show()
share = df[SPEND].var() / df[SPEND].var().sum()
print("原始尺度下，各欄佔平方距離的比例：", share.round(3).to_dict())

In [ ]:
# 相關與 Channel
# 觀察：（哪幾欄高度相關？它們代表什麼樣的客戶？Horeca 與 Retail 在哪幾欄差最多？）
L = np.log1p(df[SPEND])
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.heatmap(L.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("correlation of log spending")
L.assign(Channel=df["Channel"].map({1: "Horeca", 2: "Retail"})).melt(id_vars="Channel").pipe(
    lambda d: sns.boxplot(data=d, x="variable", y="value", hue="Channel", ax=axes[1])
)
axes[1].set_ylabel("log1p(spending)")
axes[1].tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 尺度與偏態：
#     (a) 原始尺度下，哪一欄佔平方距離最多？K-Means 用的是平方距離，不處理的話分群主要在看什麼？
#     (b) 本章的前處理是「log1p 再標準化」。log 處理了什麼問題？標準化又處理了什麼問題？（對照第四章 TODO 11）
#     (c) 有幾個點在 log 之後仍然離其他點很遠（例如某些欄位接近 0）。K-Means 對這種點敏感嗎？（提示：中心是「平均」）
# 答：
#
# (2) Milk、Grocery、Detergents_Paper 高度相關。它們一起變大的客戶，比較可能是 Horeca 還是 Retail？
#     如果分群結果大致跟 Channel 一樣，代表 K-Means 找到了什麼？
# 答：

---
## 2. 前處理與 Baseline

#### 前處理

`X = StandardScaler().fit_transform(np.log1p(df[SPEND]))`：6 欄、每欄平均 0、標準差 1。沒有訓練／測試切分（非監督學習，見開頭的表）。

#### Baseline（都用同一個 inertia 衡量）

- **全部一群**（$K = 1$）：中心是全體平均，inertia = 總平方和 $= n \cdot d$（標準化後每欄變異數 1）。任何分群都只能比它小。
- **隨機分 $K$ 群**：每個點隨機給一個群號，中心取各群平均。
- **用 Channel 分 2 群**：外部標籤直接當分群。

#### 本節名詞

- **inertia（群內平方和）**：越小代表每一群越「緊」；但 $K$ 越大一定越小（$K = n$ 時是 0），**不能直接拿來比較不同的 $K$**。
- **`clustering_metrics(X, labels, y_true)`**（`ml_dl_llm.metrics`）：silhouette（越高越好）、Davies–Bouldin（越低越好）、Calinski–Harabasz（越高越好）；給了 `y_true` 再加 ARI、NMI。

In [ ]:
from sklearn.preprocessing import StandardScaler
from ml_dl_llm.metrics import clustering_metrics

X = StandardScaler().fit_transform(np.log1p(df[SPEND]))
channel = df["Channel"].to_numpy()
n, d = X.shape


def inertia_of(X: np.ndarray, labels: np.ndarray) -> float:
    """給一組分群標籤：中心取各群平均，回傳群內平方和。"""
    return float(sum(((X[labels == k] - X[labels == k].mean(axis=0)) ** 2).sum() for k in np.unique(labels)))


rng = np.random.default_rng(0)
rand2 = rng.integers(0, 2, n)
baseline = pd.DataFrame(
    {
        "K=1 (all in one)": {"inertia": inertia_of(X, np.zeros(n, dtype=int))},
        "random K=2": {"inertia": inertia_of(X, rand2), **clustering_metrics(X, rand2, channel)},
        "Channel as clusters (K=2)": {"inertia": inertia_of(X, channel), **clustering_metrics(X, channel, channel)},
    }
).T
display(baseline.round(3))
print("✅ K=1 的 inertia 應 = n·d =", n * d)

In [ ]:
# TODO 2（問答）
#
# (1) 為什麼 K = 1 的 inertia 剛好是 n·d？（提示：標準化後每欄的變異數是 1；變異數 = 平均平方離差）
# 答：
#
# (2) 隨機分兩群，inertia 只比 K = 1 少一點點、silhouette 接近 0。為什麼？
# 答：
#
# (3) 用 Channel 分群的 inertia、silhouette 各是多少？之後 K-Means（K = 2）的 inertia 一定比它小嗎？
#     （提示：K-Means 的目標就是讓 inertia 最小）那 K-Means 分出來的兩群一定比 Channel「好」嗎？「好」是誰定義的？
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | Lloyd 演算法：分配 → 更新 → 重複；inertia | 3、4、5 |
| 3B | K-Means++ 初始化 | 6 |
| 3C | 實驗：初始化的影響、尺度與 log、K-Means 看不到的形狀 | — |

### 3A. Lloyd 演算法

目標：
$$J(c, \mu) = \sum_{i=1}^{n} \lVert x_i - \mu_{c_i}\rVert^2$$

坐標下降（前置複習 3），兩步輪流：

1. **分配（assignment）**：固定中心，每個點歸給最近的中心 $c_i = \arg\min_k \lVert x_i - \mu_k\rVert^2$（TODO 3）。對 $c$ 來說這是最好的選擇。
2. **更新（update）**：固定分配，每個中心移到它那群的平均 $\mu_k = \text{mean}\{x_i : c_i = k\}$（TODO 4）。對 $\mu$ 來說這是最好的選擇（前置複習 2）。

兩步都不會讓 $J$ 變大，而且分配方式只有有限多種 → 一定會在有限步內停下（分配不再改變）。

**空群**：如果某一群一個點都沒分到，平均沒有定義。sklearn 的做法是把它移到「離自己中心最遠的點」；這裡也照做（已寫好）。

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $X$ | 資料 | $(n, d)$ | `X` |
| $C$ | 目前的中心 | $(K, d)$ | `C`（fit 後 `self.cluster_centers_`） |
| $D$ | 每個點到每個中心的平方距離 | $(n, K)$ | `D` |
| labels | 每個點的群號 | $(n,)$ | `labels`（fit 後 `self.labels_`） |
| $J$ | inertia | 純量 | `self.inertia_` |
| history | 每一輪更新後的 $J$ | list | `self.inertia_history_` |

**下標公式**：
- TODO 3：$D_{ik} = \lVert x_i\rVert^2 + \lVert \mu_k\rVert^2 - 2 x_i \cdot \mu_k$，$c_i = \arg\min_k D_{ik}$。
- TODO 4：$\mu_{kj} = \frac{\sum_i [c_i = k] X_{ij}}{\sum_i [c_i = k]}$。
- TODO 5：$J = \sum_i \sum_j (X_{ij} - \mu_{c_i, j})^2$ → `((X - C[labels])**2).sum()`（前置複習 10）。

#### 程式結構圖

```
km = KMeansScratch(n_clusters=3, init="k-means++", max_iter=300, random_state=0).fit(X)
├── 超參數：n_clusters、init（"k-means++"／"random"／(K, d) 陣列）、max_iter、random_state
├── fit：C = _init_centers(X) (TODO 6) → 迴圈 { labels = _assign(X, C) (TODO 3)；C_new = _update(X, labels) (TODO 4)；
│                                              記錄 _inertia(X, labels, C_new) (TODO 5)；C_new == C 就停 }
└── fit 後：cluster_centers_ (K, d)、labels_ (n,)、inertia_、n_iter_、inertia_history_
```

#### 最小例子：TODO 3、4、5（一輪 Lloyd）

一維、5 個點 $x = (0, 1, 2, 8, 10)$，$K = 2$，初始中心 $\mu = (0, 2)$。

1. **正確答案**：手算。分配：0→0、1→0（距 0 為 1、距 2 為 1，平手取編號小的）、2→1、8→1、10→1；更新：$\mu_0 = 0.5$、$\mu_1 = 20/3 \approx 6.67$；inertia $= 0.25 + 0.25 + 21.78 + 1.78 + 11.11 = 35.17$。第二輪：2 離 0.5 比較近 → 換群；最後收斂到 $\mu = (1, 9)$、inertia $= 1 + 0 + 1 + 1 + 1 = 4$。
2. **積木**：距離矩陣、`argmin`（平手取第一個）、`X[labels == k].mean(axis=0)`、`C[labels]`。
3. **留給你**：寫進 `_assign`、`_update`、`_inertia`。

In [ ]:
# 最小例子：TODO 3、4、5（一輪 Lloyd）
X_mini = np.array([[0.0], [1.0], [2.0], [8.0], [10.0]])
C_mini = np.array([[0.0], [2.0]])

# ---- 1. 正確答案：迴圈 ----
C_loop = C_mini.copy()
for it in range(5):
    lab_loop = [min(range(2), key=lambda k: (x[0] - C_loop[k, 0]) ** 2) for x in X_mini]  # min 平手取第一個
    C_new = np.array([[np.mean([X_mini[i, 0] for i in range(5) if lab_loop[i] == k])] for k in range(2)])
    J_loop = sum((X_mini[i, 0] - C_new[lab_loop[i], 0]) ** 2 for i in range(5))
    print(f"第 {it + 1} 輪：labels = {lab_loop}  中心 = {C_new.ravel().round(3)}  inertia = {J_loop:.3f}")
    if np.array_equal(C_new, C_loop):
        break
    C_loop = C_new

# ---- 2. 積木 ----
D_mini = (X_mini**2).sum(axis=1)[:, None] + (C_mini**2).sum(axis=1)[None, :] - 2 * X_mini @ C_mini.T
print("\nD（第一輪）=\n", D_mini, "\nD.argmin(axis=1) =", D_mini.argmin(axis=1))
lab_mini = D_mini.argmin(axis=1)
print("X[labels == 1].mean(axis=0) =", X_mini[lab_mini == 1].mean(axis=0))
print("C[labels]（每個點自己的中心）=", C_mini[lab_mini].ravel())

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應收斂到中心 [1, 9]、inertia 4）：
# km_mini = KMeansScratch(n_clusters=2, init=C_mini).fit(X_mini); print(km_mini.cluster_centers_.ravel(), km_mini.inertia_, km_mini.inertia_history_)

In [ ]:
from typing import Literal, Self

from numpy.typing import ArrayLike


class KMeansScratch:
    """K-Means（Lloyd 演算法）。init："k-means++"、"random"，或直接給 (K, d) 的初始中心。"""

    def __init__(self, n_clusters: int = 3, init: Literal["k-means++", "random"] | np.ndarray = "k-means++", max_iter: int = 300, random_state: int | None = 0) -> None:
        self.n_clusters = n_clusters
        self.init = init
        self.max_iter = max_iter
        self.random_state = random_state

    @staticmethod
    def _assign(X: np.ndarray, C: np.ndarray) -> np.ndarray:
        """(n,)：每個點最近的中心的編號。"""
        # TODO 3（程式）：分配
        #   - 平方距離矩陣 D (n, K)：第四章的展開式（前置複習 7）
        #   - 每列取最小值的位置 → argmin(axis=1)（平手時取編號小的）
        raise NotImplementedError("TODO 3：分配")

    def _update(self, X: np.ndarray, labels: np.ndarray, C_old: np.ndarray) -> np.ndarray:
        """(K, d)：每一群的平均；空群移到「離自己中心最遠的點」（sklearn 的做法）。"""
        K = self.n_clusters
        counts = np.bincount(labels, minlength=K)
        C = C_old.copy()
        for k in range(K):
            if counts[k] > 0:
                # TODO 4（程式）：第 k 群的中心 = 這群點的平均
                raise NotImplementedError("TODO 4：更新中心")
        for k in np.flatnonzero(counts == 0):  # 空群
            far = int(((X - C_old[labels]) ** 2).sum(axis=1).argmax())
            C[k] = X[far]
        return C

    @staticmethod
    def _inertia(X: np.ndarray, labels: np.ndarray, C: np.ndarray) -> float:
        """群內平方和 J。"""
        # TODO 5（程式）：Σ_i ‖x_i − μ_{c_i}‖²；用 C[labels] 取出每個點自己的中心
        raise NotImplementedError("TODO 5：inertia")

    def _init_centers(self, X: np.ndarray) -> np.ndarray:
        if isinstance(self.init, np.ndarray):
            return self.init.astype(float).copy()
        rng = np.random.default_rng(self.random_state)
        if self.init == "random":
            return X[rng.choice(len(X), self.n_clusters, replace=False)].copy()
        return kmeans_plusplus_init(X, self.n_clusters, rng)

    def fit(self, X: ArrayLike) -> Self:
        X = np.asarray(X, dtype=float)
        C = self._init_centers(X)
        self.inertia_history_ = []
        for it in range(1, self.max_iter + 1):
            labels = self._assign(X, C)
            C_new = self._update(X, labels, C)
            self.inertia_history_.append(self._inertia(X, labels, C_new))
            if np.array_equal(C_new, C):  # 中心不再移動 → 分配也不會再變
                break
            C = C_new
        self.cluster_centers_ = C_new
        self.labels_ = self._assign(X, C_new)
        self.inertia_ = self._inertia(X, self.labels_, C_new)
        self.n_iter_ = it
        return self

    def predict(self, X: ArrayLike) -> np.ndarray:
        return self._assign(np.asarray(X, dtype=float), self.cluster_centers_)

### 3B. K-Means++ 初始化

Lloyd 演算法只保證停在「調不動」的地方（前置複習 3）。初始中心如果剛好擠在同一團裡，可能永遠分不開別的團。**K-Means++**（Arthur & Vassilvitskii 2007）讓初始中心盡量分散：

1. 第一個中心：隨機挑一個資料點；
2. 之後每一個中心：算每個點到「目前最近的中心」的平方距離 $D(x)^2$，**以正比於 $D(x)^2$ 的機率**挑下一個中心（前置複習 4）—— 離現有中心越遠的點，越可能被挑中；已經是中心的點 $D = 0$，不會再被挑。

理論保證：K-Means++ 初始化的期望 inertia 不超過最佳解的 $O(\log K)$ 倍。

（sklearn 的版本每一步會多抽幾個候選、挑讓 inertia 最小的那個 —— "greedy k-means++"，所以跟這裡不會完全一樣。）

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| centers | 已經挑好的中心 | $(m, d)$，$m$ 從 1 長到 $K$ | `centers` |
| $D(x)^2$ | 每個點到最近中心的平方距離 | $(n,)$ | `d2` |
| $p$ | 抽樣機率 $D(x)^2 / \sum D^2$ | $(n,)$ | `p` |

#### 最小例子：TODO 6（D² 抽樣）

一維 $x = (0, 1, 2, 8, 10)$，第一個中心已經挑了 $x = 0$。

1. **正確答案**：$D^2 = (0, 1, 4, 64, 100)$，總和 169 → 機率 $(0, 0.006, 0.024, 0.379, 0.592)$：下一個中心有 97% 的機率落在右邊那團。
2. **積木**：到每個中心的平方距離矩陣、`min(axis=1)`、正規化、`rng.choice(n, p=p)`。
3. **留給你**：算出 `p`。

In [ ]:
# 最小例子：TODO 6（D² 抽樣）
centers_mini = np.array([[0.0]])

# ---- 1. 正確答案 ----
d2_loop = [min((x[0] - c[0]) ** 2 for c in centers_mini) for x in X_mini]
print("D² =", d2_loop, "→ p =", np.round(np.array(d2_loop) / sum(d2_loop), 3))

# ---- 2. 積木 ----
D_c = (X_mini**2).sum(axis=1)[:, None] + (centers_mini**2).sum(axis=1)[None, :] - 2 * X_mini @ centers_mini.T
print("\n到每個中心的平方距離 (n, m) =", D_c.ravel(), "；.min(axis=1) =", D_c.min(axis=1))
rng_mini = np.random.default_rng(0)
print("rng.choice(5, p=[0, 0, 0, 0.5, 0.5]) 抽 10 次：", [int(rng_mini.choice(5, p=[0, 0, 0, 0.5, 0.5])) for _ in range(10)])

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（K = 2：兩個中心幾乎一定是「一個在左團 {0, 1, 2}、一個在右團 {8, 10}」）：
# print([kmeans_plusplus_init(X_mini, 2, np.random.default_rng(s)).ravel().tolist() for s in range(6)])

In [ ]:
def kmeans_plusplus_init(X: np.ndarray, K: int, rng: np.random.Generator) -> np.ndarray:
    """K-Means++：回傳 (K, d) 的初始中心（都是資料點）。"""
    n = len(X)
    centers = [X[rng.integers(n)]]
    for _ in range(1, K):
        C = np.array(centers)
        D = np.maximum((X**2).sum(axis=1)[:, None] + (C**2).sum(axis=1)[None, :] - 2 * X @ C.T, 0)  # (n, m)
        # TODO 6（程式）：D² 抽樣的機率
        #   - d2 = 每個點到最近中心的平方距離（D 每列的最小值）
        #   - p = d2 / d2 的總和
        raise NotImplementedError("TODO 6：K-Means++")
        centers.append(X[rng.choice(n, p=p)])
    return np.array(centers)


from sklearn.cluster import KMeans

# 檢查 1：同樣的初始中心 → 跟 sklearn 的 Lloyd 完全一樣
rng_c = np.random.default_rng(0)
for K in [2, 3, 4, 6]:
    C0 = X[rng_c.choice(n, K, replace=False)]
    mine = KMeansScratch(n_clusters=K, init=C0).fit(X)
    sk = KMeans(n_clusters=K, init=C0, n_init=1, algorithm="lloyd", tol=0).fit(X)
    print(
        f"K={K}：✅ labels 一致 {np.mean(mine.labels_ == sk.labels_):.3f}（應 = 1.0）",
        f"；中心最大差異 {np.abs(mine.cluster_centers_ - sk.cluster_centers_).max():.1e}（應 < 1e-10）",
        f"；inertia {mine.inertia_:.3f} vs {sk.inertia_:.3f}；{mine.n_iter_} 輪",
    )

# 檢查 2：inertia 每一輪都不會變大
km3 = KMeansScratch(n_clusters=3, init="random", random_state=1).fit(X)
print("✅ inertia_history_ 單調不增：", bool(np.all(np.diff(km3.inertia_history_) <= 1e-9)), "；", np.round(km3.inertia_history_, 2))

In [ ]:
# TODO 7（問答）Lloyd 演算法
#
# (1) 用前置複習 2、3 證明：分配與更新兩步都不會讓 J 變大。為什麼這保證演算法「有限步內停下」？
#     （提示：n 個點分成 K 群的方式有幾種？同一種分配不會出現兩次 —— 為什麼？）
# 答：
#
# (2) 最小例子：第一輪中心是 (0.5, 6.67)，最後收斂到 (1, 9)。哪一個點在第二輪「換群」了？為什麼？
# 答：
#
# (3) 跟第四章的 NearestCentroid 比：兩者的「預測」都是「找最近的中心」。差別在中心怎麼來？
#     K-Means 的群在空間中被切成什麼形狀？（提示：兩個中心之間的分界是什麼？—— 第四章 TODO 2 (2)）
# 答：
#
# (4) 一輪的成本是 O(?)（用 n、K、d 表示）？跟第四章 KNN 預測一筆要比 n 個點比，K-Means 預測一筆要比幾個？
# 答：

### 3C. 實驗：初始化、尺度、K-Means 看不到的形狀

#### 本節名詞

| 名詞 | 定義 |
| --- | --- |
| 局部最小 | Lloyd 停下的地方：分配與中心互相「最好」，但換一個初始值可能找到 inertia 更小的解 |
| `n_init` | sklearn 會用不同的初始值跑 `n_init` 次，留下 inertia 最小的那一次 |
| 非等向（anisotropic） | 一群點在某個方向特別長（橢圓、斜的） |
| 非凸（non-convex） | 例如兩個互相勾住的半月形（`make_moons`） |

In [ ]:
# 實驗 1：初始化。random vs k-means++，各跑 200 次，看最後的 inertia 分布
# (a) 本章資料（K = 4）；(b) 合成資料：10 團分得很開的點（K = 10）
from sklearn.datasets import make_blobs

X_sep, _ = make_blobs(n_samples=1000, centers=10, cluster_std=0.5, center_box=(-15, 15), random_state=3)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
rows = []
for ax, (data_name, Xd, K) in zip(axes, [("wholesale (K=4)", X, 4), ("10 well-separated blobs (K=10)", X_sep, 10)]):
    best_known = KMeans(K, n_init=50, random_state=0).fit(Xd).inertia_
    for init in ["random", "k-means++"]:
        v = np.array([KMeansScratch(n_clusters=K, init=init, random_state=s).fit(Xd).inertia_ for s in range(200)])
        ax.hist(v / best_known, bins=40, alpha=0.6, label=init)
        rows.append({"data": data_name, "init": init, "mean / best": v.mean() / best_known, "worst / best": v.max() / best_known, "在最佳 1% 內的比例": np.mean(v <= best_known * 1.01)})
    ax.set_xlabel("final inertia / best known")
    ax.set_title(data_name)
    ax.legend()
plt.show()
display(pd.DataFrame(rows).set_index(["data", "init"]).round(3))

In [ ]:
# 實驗 2：前處理。raw / 只標準化 / 只取 log / log + 標準化，K = 2，看分群跟 Channel 的一致程度與群的大小
variants = {
    "raw": df[SPEND].to_numpy(dtype=float),
    "standardized only": StandardScaler().fit_transform(df[SPEND]),
    "log1p only": np.log1p(df[SPEND]).to_numpy(),
    "log1p + standardized": X,
}
rows = []
for name, Xv in variants.items():
    lab = KMeans(2, n_init=10, random_state=0).fit_predict(Xv)
    m = clustering_metrics(Xv, lab, channel)
    rows.append({"preprocessing": name, "cluster sizes": sorted(np.bincount(lab).tolist()), "ARI vs Channel": m["ari"], "silhouette (in its own space)": m["silhouette"]})
display(pd.DataFrame(rows).set_index("preprocessing"))

In [ ]:
# 實驗 3：K-Means 看不到的形狀（合成資料）
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score

Xa, ya = make_blobs(n_samples=600, centers=3, random_state=170)
Xa = Xa @ np.array([[0.6, -0.64], [-0.4, 0.85]])  # 拉成斜的橢圓
Xb, yb = make_blobs(n_samples=600, centers=3, cluster_std=[1.0, 2.5, 0.5], random_state=170)
Xc, yc = make_moons(n_samples=600, noise=0.05, random_state=0)
Xd, yd = make_blobs(n_samples=[500, 50, 50], centers=[[0, 0], [5, 5], [-5, 5]], cluster_std=[2.0, 0.5, 0.5], random_state=0)
cases = {"anisotropic": (Xa, ya, 3), "unequal variance": (Xb, yb, 3), "non-convex (moons)": (Xc, yc, 2), "unequal sizes": (Xd, yd, 3)}
fig, axes = plt.subplots(1, 4, figsize=(19, 4))
for ax, (name, (Xs, ys, K)) in zip(axes, cases.items()):
    lab = KMeans(K, n_init=10, random_state=0).fit_predict(Xs)
    ax.scatter(Xs[:, 0], Xs[:, 1], c=lab, cmap="viridis", s=6)
    ax.set_title(f"{name}\nARI vs truth = {adjusted_rand_score(ys, lab):.2f}")
plt.show()

In [ ]:
# TODO 8（問答）初始化、前處理、形狀
#
# (1) 實驗 1：
#     (a) 兩份資料上，random 與 k-means++ 的「在最佳 1% 內的比例」各是多少？哪一份資料上 k-means++ 的優勢明顯？
#     (b) 為什麼「初始中心擠在一起」會讓 Lloyd 卡住？（提示：10 團分得很開時，兩個中心一開始落在同一團，另一團就沒有中心了 ——
#         一個中心要怎麼「跨過」中間的空白過去？）
#     (c) 本章資料（silhouette 只有 0.2–0.3）上兩者差不多，而且很多次的結果都在最佳的 1% 內。這說明這份資料的「群」是什麼樣子？
#     (d) sklearn 預設 n_init="auto"（k-means++ 時跑 1 次）。看實驗 1 的結果，你會建議調大嗎？代價是什麼？
# 答：
#
# (2) 實驗 2：
#     (a) raw 的兩群大小是多少？它主要依哪一欄在分？（TODO 1 (1)(a)）
#     (b) 哪一種前處理跟 Channel 最一致？「log 但不標準化」為什麼也還不錯？（提示：log 之後各欄的尺度差多少？）
#     (c) 這說明了：對 K-Means，「前處理」其實是在定義什麼？（第四章 TODO 11 的同一句話）
# 答：
#
# (3) 實驗 3：每一種情況，K-Means 錯在哪裡？用「K-Means 的群一定是哪種形狀」（TODO 7 (3)）解釋。
#     哪幾種在下兩章（DBSCAN：依密度；GMM：每群一個橢圓）會比較好處理？
# 答：

---
## 4. 套件實作：sklearn `KMeans`、`MiniBatchKMeans`

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| `KMeans(algorithm="lloyd")` | 本章的演算法；`"elkan"` 用三角不等式跳過不必要的距離計算（結果一樣、可能比較快） |
| `n_init` | 跑幾次不同的初始化，留最好的 |
| `tol` | 中心移動量小於多少就停（sklearn 預設 $10^{-4}$，所以可能比「完全不動」早停） |
| `MiniBatchKMeans` | 每一步只用一小批隨機樣本更新中心（線上平均）；大資料時快很多，inertia 稍差 |
| `transform(X)` | 每個點到每個中心的距離 $(n, K)$ —— 也可以當成新特徵 |

In [ ]:
import time

from sklearn.cluster import MiniBatchKMeans

# 一份大一點的合成資料比較速度
X_big, _ = make_blobs(n_samples=200_000, centers=8, n_features=6, random_state=0)
rows = []
for name, make in {
    "KMeans lloyd": lambda: KMeans(8, n_init=1, algorithm="lloyd", random_state=0),
    "KMeans elkan": lambda: KMeans(8, n_init=1, algorithm="elkan", random_state=0),
    "MiniBatchKMeans (batch 1024)": lambda: MiniBatchKMeans(8, n_init=1, batch_size=1024, random_state=0),
    "scratch (k-means++)": lambda: KMeansScratch(8, random_state=0),
}.items():
    t0 = time.perf_counter()
    m = make().fit(X_big)
    rows.append({"model": name, "fit s": time.perf_counter() - t0, "inertia": m.inertia_})
speed = pd.DataFrame(rows).set_index("model")
speed["inertia vs best"] = speed["inertia"] / speed["inertia"].min()
display(speed.round(3))

km_sk = KMeans(4, n_init=10, random_state=0).fit(X)
print("本章資料 K = 4：sklearn inertia", round(km_sk.inertia_, 3), "；transform(X) shape", km_sk.transform(X).shape)

In [ ]:
# TODO 9（問答）
#
# (1) 在 20 萬筆的合成資料上，MiniBatchKMeans 有比 Lloyd 快嗎？inertia 差多少（%）？
#     （sklearn 的 Lloyd 已經高度最佳化、資料也放得進記憶體）什麼情況下 MiniBatch 才值得用？（資料大到放不進記憶體、或資料一直進來）
# 答：
#
# (2) KMeans.transform(X) 給每個點到 K 個中心的距離。如果把它當成新特徵丟進監督模型（例如第三章的邏輯回歸），
#     它在做的事跟第八章的 RBF kernel 有什麼相似？
# 答：

---
## 5. 選群數 $K$

非監督學習沒有「正確答案」可以 CV。常用的線索：

| 方法 | 看什麼 | 缺點 |
| --- | --- | --- |
| **elbow** | inertia 對 $K$ 的曲線，找「轉折」 | inertia 一定隨 $K$ 下降；轉折常常不明顯 |
| **silhouette** | $s_i = \frac{b_i - a_i}{\max(a_i, b_i)}$ 的平均；$a_i$：$i$ 到自己群其他點的平均距離，$b_i$：$i$ 到「最近的別群」所有點的平均距離 | 偏好凸、分得開的群；$O(n^2)$ |
| **穩定度** | 對資料重抽樣、重跑分群，兩次結果的 ARI；好的 $K$ 應該很穩定 | 要跑很多次 |
| **可解釋性** | 每一群的中心有沒有商業意義 | 主觀，但常常最重要 |

#### 符號表（silhouette）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $D$ | 點與點之間的**歐氏距離**矩陣（不是平方） | $(n, n)$ | `D` |
| $a_i$ | $i$ 到同群其他點的平均距離（不含自己）；只有自己一個點的群定義 $s_i = 0$ | $(n,)$ | `a` |
| $\text{mean}_{ik}$ | $i$ 到第 $k$ 群所有點的平均距離 | $(n, K)$ | `M` |
| $b_i$ | $\min_{k \ne c_i} \text{mean}_{ik}$ | $(n,)$ | `b` |

#### 最小例子：TODO 10（silhouette）

一維 $x = (0, 1, 2, 8, 10)$，分群 $c = (0, 0, 0, 1, 1)$。

1. **正確答案**：迴圈。例：$x = 0$：$a = (1 + 2)/2 = 1.5$、$b = (8 + 10)/2 = 9$ → $s = 7.5/9 = 0.833$。全部平均 0.797（與 sklearn 相同）。
2. **積木**：距離矩陣 `D`、每一群的平均距離 `M[:, k] = D[:, c == k].mean(axis=1)`、自己那一群要扣掉自己（距離 0）再除以 $n_k - 1$。
3. **留給你**：組出 `a`、`b`、`s`。

In [ ]:
# 最小例子：TODO 10（silhouette）
from sklearn.metrics import silhouette_samples, silhouette_score

c_mini = np.array([0, 0, 0, 1, 1])
x1 = X_mini.ravel()

# ---- 1. 正確答案：迴圈 ----
s_loop = []
for i in range(5):
    same = [abs(x1[i] - x1[j]) for j in range(5) if c_mini[j] == c_mini[i] and j != i]
    other = [abs(x1[i] - x1[j]) for j in range(5) if c_mini[j] != c_mini[i]]
    a_i, b_i = np.mean(same), np.mean(other)
    s_loop.append((b_i - a_i) / max(a_i, b_i))
print("正確答案 s =", np.round(s_loop, 4), " 平均", round(float(np.mean(s_loop)), 4), "；sklearn", round(silhouette_score(X_mini, c_mini), 4))

# ---- 2. 積木 ----
D_s = np.abs(x1[:, None] - x1[None, :])
M_s = np.column_stack([D_s[:, c_mini == k].mean(axis=1) for k in range(2)])
print("\nM[:, k] = D[:, c == k].mean(axis=1)（含自己的 0）=\n", M_s.round(3))
n_k = np.bincount(c_mini)
print("自己那一群：M[i, c_i] · n_k / (n_k − 1) 才是「不含自己」的平均 →", (M_s[np.arange(5), c_mini] * n_k[c_mini] / (n_k[c_mini] - 1)).round(3))

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(silhouette_scratch(X_mini, c_mini).round(4))

In [ ]:
def silhouette_scratch(X: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """(n,)：每個點的 silhouette。只有自己一個點的群給 0。"""
    X = np.asarray(X, dtype=float)
    D = np.sqrt(np.maximum((X**2).sum(axis=1)[:, None] + (X**2).sum(axis=1)[None, :] - 2 * X @ X.T, 0))
    K = labels.max() + 1
    n_k = np.bincount(labels, minlength=K)
    M = np.column_stack([D[:, labels == k].mean(axis=1) for k in range(K)])  # (n, K)，自己那一群含自己的 0
    rows = np.arange(len(X))
    # TODO 10（程式）
    #   - a = M[rows, labels] · n_k[labels] / (n_k[labels] − 1)       （扣掉自己；n_k = 1 時先不管，最後設成 0）
    #   - b = M 裡「不是自己那一群」的最小值：先把 M[rows, labels] 設成 inf 再 min(axis=1)（用 M.copy()）
    #   - s = (b − a) / max(a, b)；n_k[labels] == 1 的點 s = 0
    raise NotImplementedError("TODO 10：silhouette")


lab4 = KMeans(4, n_init=10, random_state=0).fit_predict(X)
print("✅ 與 sklearn silhouette_samples 的最大差異：", np.abs(silhouette_scratch(X, lab4) - silhouette_samples(X, lab4)).max(), "（應 < 1e-8）")

In [ ]:
# 選 K：elbow、silhouette、穩定度、跟 Channel 的一致程度
Ks = range(2, 11)
rows = []
for K in Ks:
    km = KMeans(K, n_init=10, random_state=0).fit(X)
    # 穩定度：20 次「抽 80% 的資料重跑」，跟全資料的分群在那 80% 上比 ARI
    rng_s = np.random.default_rng(K)
    stab = []
    for _ in range(20):
        sub = rng_s.choice(n, int(0.8 * n), replace=False)
        lab_sub = KMeans(K, n_init=10, random_state=int(rng_s.integers(1e6))).fit_predict(X[sub])
        stab.append(adjusted_rand_score(km.labels_[sub], lab_sub))
    rows.append({"K": K, "inertia": km.inertia_, "silhouette": silhouette_scratch(X, km.labels_).mean(), "stability (ARI)": np.mean(stab), "ARI vs Channel": adjusted_rand_score(channel, km.labels_)})
k_tbl = pd.DataFrame(rows).set_index("K")
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
axes[0].plot(k_tbl.index, k_tbl["inertia"], "o-")
axes[0].set_title("Elbow: inertia")
axes[1].plot(k_tbl.index, k_tbl["silhouette"], "o-", color="C1")
axes[1].set_title("Mean silhouette")
axes[2].plot(k_tbl.index, k_tbl["stability (ARI)"], "o-", color="C2")
axes[2].set_title("Stability under 80% subsampling")
for ax in axes:
    ax.set_xlabel("K")
fig.tight_layout()
plt.show()
display(k_tbl.round(3))

# K = 2、3、4 的 silhouette 圖：每一群的 s_i 排序後畫出來
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, K in zip(axes, [2, 3, 4]):
    lab = KMeans(K, n_init=10, random_state=0).fit_predict(X)
    s = silhouette_scratch(X, lab)
    y0 = 0
    for k in range(K):
        sk_ = np.sort(s[lab == k])
        ax.fill_betweenx(np.arange(y0, y0 + len(sk_)), 0, sk_, alpha=0.7)
        y0 += len(sk_) + 10
    ax.axvline(s.mean(), color="r", ls="--")
    ax.set_title(f"K = {K}: mean silhouette {s.mean():.3f}")
    ax.set_xlabel("silhouette")
plt.show()

In [ ]:
# TODO 11（問答）選 K
#
# (1) elbow：曲線有明顯的轉折嗎？如果沒有，代表資料是什麼樣子？
# 答：
#
# (2) silhouette：哪個 K 最高？數值多少？（經驗上 > 0.5 代表結構很清楚、0.25–0.5 普通、< 0.25 很弱）
#     看 silhouette 圖：K = 4 時有沒有群的很多點 s < 0？那代表什麼？
# 答：
#
# (3) 穩定度：哪個 K 最穩？K 大時為什麼比較不穩？
# 答：
#
# (4) 綜合四個線索，你會選幾群？如果業務單位說「我們想把客戶分成 4–5 種來設計不同的方案」，你的回答會改變嗎？
#     「選 K」有沒有唯一正確答案？
# 答：

---
## 6. 評估

沒有測試集；用同一份資料算**內部指標**（silhouette、Davies–Bouldin、Calinski–Harabasz）與**外部指標**（跟 Channel 比的 ARI、NMI —— Channel 在分群時完全沒用到）。

- `clustering_metrics(X, labels, y_true=channel)`；`log_result`／`compare` 寫進 `results/wholesale.csv`，依 silhouette 排序。
- 對照：隨機分群、用 Channel 當分群、以及第 12、13 章之後可以回來加的 DBSCAN、GMM。

In [ ]:
from ml_dl_llm.results import compare, log_result

BENCH = "wholesale"
K_CHOSEN = int(k_tbl["silhouette"].idxmax())
eval_runs = {
    "Random labels K=2": (rand2, {}),
    "Channel as clusters": (channel, {}),
    f"KMeans K={K_CHOSEN} (scratch)": (KMeansScratch(K_CHOSEN, random_state=0).fit(X).labels_, {"K": K_CHOSEN}),
    f"KMeans K={K_CHOSEN} (sklearn)": (KMeans(K_CHOSEN, n_init=10, random_state=0).fit_predict(X), {"K": K_CHOSEN, "n_init": 10}),
    "KMeans K=4 (sklearn)": (KMeans(4, n_init=10, random_state=0).fit_predict(X), {"K": 4, "n_init": 10}),
    "KMeans K=2 on raw spending (sklearn)": (KMeans(2, n_init=10, random_state=0).fit_predict(df[SPEND]), {"K": 2, "preprocessing": "raw"}),
}
scores = {}
for name, (lab, params) in eval_runs.items():
    m = clustering_metrics(X, lab, channel)  # 一律在 log+標準化 的空間裡算內部指標，才能比較
    scores[name] = m
    log_result(BENCH, name, m, **params)
display(pd.DataFrame(scores).T)
compare(BENCH, "silhouette")

---
## 7. 診斷：每一群是什麼樣的客戶？

把群中心換回**原始金額**（中位數比較穩），再跟 Channel、Region 交叉比對；用 PCA（第 16 章）投影到 2 維畫圖。

In [ ]:
from sklearn.decomposition import PCA

K_SHOW = 4
km_final = KMeans(K_SHOW, n_init=10, random_state=0).fit(X)
lab = km_final.labels_
profile = df[SPEND].groupby(lab).median().round(0)
profile["size"] = np.bincount(lab)
profile["% Retail"] = (pd.Series(channel == 2).groupby(lab).mean() * 100).round(1)
print(f"K = {K_SHOW}：各群的消費中位數（原始金額）")
display(profile)
display(pd.crosstab(lab, df["Region"], normalize="index").round(2).rename_axis(index="cluster", columns="Region"))

Z = PCA(n_components=2).fit(X)
Z2 = Z.transform(X)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(Z2[:, 0], Z2[:, 1], c=lab, cmap="tab10", s=12)
C2 = Z.transform(km_final.cluster_centers_)
axes[0].scatter(C2[:, 0], C2[:, 1], c="k", marker="X", s=150)
axes[0].set_title(f"K-Means (K = {K_SHOW}) in PCA 2D")
axes[1].scatter(Z2[:, 0], Z2[:, 1], c=channel, cmap="coolwarm", s=12)
axes[1].set_title("Channel (1 = Horeca, 2 = Retail)")
for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.show()

# 離自己中心最遠的點：可能的離群客戶
dist = np.sqrt(((X - km_final.cluster_centers_[lab]) ** 2).sum(axis=1))
far = np.argsort(dist)[::-1][:5]
display(df.iloc[far].assign(cluster=lab[far], distance=dist[far].round(2)))

In [ ]:
# TODO 12（問答，本章總結）
#
# (1) 每一群是什麼樣的客戶？給每一群取一個名字（例如「大量採購雜貨的零售商」），並說明根據（看中位數表）。
#     各群的 % Retail 是多少？哪幾群幾乎都是 Horeca 或幾乎都是 Retail？Region 有差嗎？
# 答：
#
# (2) PCA 圖：K-Means 的群在圖上是不是一塊一塊、彼此用「直線」分開？Channel 的兩類呢？
#     2 維投影上的重疊，能代表 6 維空間也重疊嗎？（第四章 TODO 1 (4)）
# 答：
#
# (3) 離中心最遠的幾個客戶有什麼特徵（看原始金額）？它們把自己那一群的中心「拉」走了多少？
#     如果要讓分群對這種點比較不敏感，可以怎麼做？（提示：中心用中位數 → K-Medians／K-Medoids；或下一章 DBSCAN 把它們當雜訊）
# 答：
#
# (4) 總結 K-Means 的假設與限制：它最適合什麼樣的群？（大小、形狀、密度）這份資料符合嗎？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

Wholesale Customers（440 筆、6 個消費欄位；前處理 log1p + 標準化；Channel 只用於事後檢查）。

| 方法 | 做法 | 關鍵性質 | silhouette | ARI vs Channel |
| --- | --- | --- | --- | --- |
| 隨機分群 | | | | |
| Channel 當分群 | | | | |
| K-Means（手寫） | Lloyd + K-Means++ | | | |
| K-Means（sklearn，選定的 K） | | | | |
| K-Means（原始金額） | | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**演算法**
- ☐ Elkan、Hamerly：用三角不等式加速；K-Means‖（平行版的 K-Means++）
- ☐ K-Medoids（PAM）、K-Medians：對離群值比較穩
- ☐ Kernel K-Means、Spectral Clustering：非凸的群
- ☐ K-Means 是 NP-hard；K-Means++ 的 $O(\log K)$ 近似保證

**選 K 與評估**
- ☐ Gap statistic、BIC（第 15 章 GMM）
- ☐ 分群穩定度的更嚴謹做法（consensus clustering）
- ☐ 內部指標各自偏好什麼樣的群（silhouette、DB、CH 的比較）

**應用**
- ☐ 向量量化（vector quantization）：影像壓縮、K-Means 當特徵（`transform`）
- ☐ 客戶分群的實務：RFM 特徵、業務可解釋性